# Build Evidence-Grounded Factoid DPO Pairs

Construct standard pairwise factoid DPO data from the evidence-only candidate banks. The chosen answer is a gold alias supported by snippets; each rejected answer is an entity-like generated candidate that is not a gold alias.


In [ ]:
from pathlib import Path
import json

cwd=Path.cwd().resolve()
candidates=[cwd,*cwd.parents,cwd/'Task-Structured Counterfactual Preference Mining']
PROJECT_ROOT=next((path for path in candidates if (path/'src').is_dir() and (path/'cse_dpo').is_dir()),None)
assert PROJECT_ROOT is not None, 'Run this notebook from the project or workspace directory.'

PREPARED_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/evidence_grounded_single_answer_qwen25_05b'
TRAIN_QUESTIONS=PREPARED_DIR/'train_prepared.json'
DEV_QUESTIONS=PREPARED_DIR/'dev_supported_prepared.json'
MODEL_SLUG='evidence-grounded-single-answer-qwen25-05b'
TRAIN_CANDIDATES=PROJECT_ROOT/'Artifacts/cse_dpo/candidate_banks/factoid_evidence_only_single_answer_qwen25_05b_train_s3_n20'/MODEL_SLUG/'candidate_bank.jsonl'
DEV_CANDIDATES=PROJECT_ROOT/'Artifacts/cse_dpo/candidate_banks/factoid_evidence_only_single_answer_qwen25_05b_dev_s3_n20'/MODEL_SLUG/'candidate_bank.jsonl'
OUTPUT_ROOT=PROJECT_ROOT/'Artifacts/cse_dpo/evidence_grounded_gold_response_pairs_qwen25_05b/run1'

for path in (TRAIN_QUESTIONS,DEV_QUESTIONS,TRAIN_CANDIDATES,DEV_CANDIDATES):
    assert path.exists(), f'Missing required input: {path}'
print(f'Train candidates: {TRAIN_CANDIDATES}')
print(f'Dev candidates: {DEV_CANDIDATES}')
print(f'Output: {OUTPUT_ROOT}')


## Construct Pairs

The factoid builder removes gold aliases from generated candidates and ranks distinct wrong entities by within-question frequency. This strict run retains only questions with all four negatives generated for that same question; it never uses global backfill.


In [ ]:
import subprocess

splits={
    'train': {'questions':TRAIN_QUESTIONS,'candidates':TRAIN_CANDIDATES},
    'dev': {'questions':DEV_QUESTIONS,'candidates':DEV_CANDIDATES},
}
for split_name, split in splits.items():
    split_dir=OUTPUT_ROOT/split_name
    command=[
        'python','-m','cse_dpo.construct_factoid_label_wrong_entity_bank',
        '--question-input',str(split['questions']),
        '--candidate-bank-jsonl',str(split['candidates']),
        '--output-jsonl',str(split_dir/'question_classes.jsonl'),
        '--pair-output-jsonl',str(split_dir/'pairs.jsonl'),
        '--summary-json',str(split_dir/'summary.json'),
        '--wrongs-per-question','4',
        '--require-within-question-wrongs',
        '--max-entity-chars','160','--max-entity-words','14',
        '--seed','3407',
    ]
    print(f'Constructing {split_name} pairs:')
    print(' '.join(command))
    subprocess.run(command,cwd=PROJECT_ROOT,check=True)


## Review Pair Quality

The preview below is the manual audit. Confirm that the chosen answer is a supported gold entity and the rejected entity is plausible but incorrect before training.


In [ ]:
for split_name in ('train','dev'):
    split_dir=OUTPUT_ROOT/split_name
    summary_path=split_dir/'summary.json'
    pairs_path=split_dir/'pairs.jsonl'
    classes_path=split_dir/'question_classes.jsonl'
    assert summary_path.exists() and pairs_path.exists() and classes_path.exists(), f'Missing {split_name} outputs.'
    summary=json.loads(summary_path.read_text(encoding='utf-8'))
    pair_count=sum(1 for line in pairs_path.open(encoding='utf-8') if line.strip())
    class_count=sum(1 for line in classes_path.open(encoding='utf-8') if line.strip())
    print(f'[{split_name}] question classes: {class_count}; pair rows: {pair_count}')
    print(json.dumps(summary,indent=2))
    with pairs_path.open(encoding='utf-8') as handle:
        preview=[json.loads(next(handle)) for _ in range(min(3,pair_count))]
    for row in preview:
        print(f"  {row['question_id']}: chosen={row['chosen']!r}; rejected={row['rejected']!r}; source={row['rejected_source']}")
